In [1]:
import numpy as np
import pandas as pd
import random
import tensorflow as tf
import keras_tuner as kt
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_score, recall_score,
    accuracy_score, f1_score, confusion_matrix, precision_recall_curve, auc, matthews_corrcoef
)
import matplotlib.pyplot as plt
import joblib
import os

# ============================
# 1. Define Datasets
# ============================
positive_files = [
    "pos_charge_gravy.xlsx"  # Change extension to .csv if needed
]

negative_files = [
    "hor_charge_gravy.xlsx",  # Exact raw/unfiltered negative file 1
    "non-AMP_charge_gravy.xlsx"   # Exact raw/unfiltered negative file 2
]

def load_dataset(file_path):
    if file_path.endswith('.csv'):
        df = pd.read_csv(file_path)
    else:
        df = pd.read_excel(file_path)
    
    seq_col = None
    for col in df.columns:
        if col.strip().lower() in ['sequence', 'seq', 'peptides', 'peptide']:
            seq_col = col
            break
            
    if seq_col is None:
        raise KeyError(f"Could not find sequence column in {file_path}. Available columns: {df.columns.tolist()}")
    
    df = df.rename(columns={seq_col: "Sequence"})
    return df[["Sequence"]].dropna()


# ============================
# 2. Encoding Setup
# ============================
amino_acids = "ACDEFGHIKLMNPQRSTVWY"
aa_to_int = {aa: i+1 for i, aa in enumerate(amino_acids)}  # 0 = padding
num_tokens = len(amino_acids) + 1
max_length = 30

def encode_sequence(seq):
    return [aa_to_int[aa] for aa in str(seq).strip().upper() if aa in aa_to_int]

def preprocess_sequences(sequences):
    encoded = [encode_sequence(seq) for seq in sequences]
    padded = tf.keras.preprocessing.sequence.pad_sequences(
        encoded,
        maxlen=max_length,
        padding="post"
    )
    return padded


# ============================
# 3. Augmentation Function
# ============================
similar_groups = [
    ['A','V','L','I'],   # aliphatic
    ['F','W','Y'],       # aromatic
    ['K','R','H'],       # positive
    ['D','E'],           # negative
    ['S','T','N','Q']    # polar
]

def mutate_seq(seq, p_mut=0.08):
    seq = list(str(seq))
    for i, aa in enumerate(seq):
        if random.random() < p_mut:
            for g in similar_groups:
                if aa in g:
                    choices = [x for x in g if x != aa]
                    if choices:
                        seq[i] = random.choice(choices)
                    break
    return "".join(seq)


# ============================
# 4. Attention Layer
# ============================
class Attention(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="random_normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)
        super().build(input_shape)

    def call(self, x):
        e = tf.keras.backend.tanh(tf.keras.backend.dot(x, self.W) + self.b)
        a = tf.keras.backend.softmax(e, axis=1)
        return tf.keras.backend.sum(x * a, axis=1)


# ============================
# 5. Model Builder
# ============================
def build_model(hp):
    inputs = tf.keras.layers.Input(shape=(max_length, num_tokens))
    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(
            units=hp.Int("units", 64, 128, step=32),
            return_sequences=True,
            dropout=hp.Float("dropout", 0.2, 0.4, step=0.1),
            recurrent_dropout=0.2
        )
    )(inputs)
    x = Attention()(x)
    x = tf.keras.layers.Dropout(hp.Float("att_dropout", 0.2, 0.4, step=0.1))(x)
    x = tf.keras.layers.Dense(
        hp.Int("dense_units", 32, 64, step=32), activation="relu"
    )(x)
    x = tf.keras.layers.Dropout(hp.Float("dense_dropout", 0.2, 0.4, step=0.1))(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    
    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(
            learning_rate=hp.Choice("lr", [0.001, 0.0005])
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


# ============================
# 6. Training Loop (All Files Saved with Prefix 'NF_')
# ============================
for pi, pos_file in enumerate(positive_files, start=1):
    for ni, neg_file in enumerate(negative_files, start=1):
        
        # Define base prefix for all outputs
        base_prefix = f"NF_model_P{pi}_N{ni}"
        
        print(f"\n=======================================================")
        print(f"[INFO] Training Classifier: {base_prefix}")
        print(f"Positive File: {pos_file}")
        print(f"Negative File: {neg_file}")
        print(f"=======================================================")

        # --- Set seeds
        base_seed = 1000 + pi*10 + ni
        tf.random.set_seed(base_seed)
        np.random.seed(base_seed)
        random.seed(base_seed)

        # --- Load datasets
        df_pos = load_dataset(pos_file)
        df_neg = load_dataset(neg_file)

        df_pos["label"] = 1
        df_neg["label"] = 0

        df_all = pd.concat([df_pos[["Sequence", "label"]],
                            df_neg[["Sequence", "label"]]], ignore_index=True)

        # --- Train-test split (80:20 stratified)
        X = df_all["Sequence"].values
        y = df_all["label"].values
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=base_seed, shuffle=True
        )

        # --- Save Raw Split Datasets Separately (with NF_ Prefix)
        df_train_raw = pd.DataFrame({"Sequence": X_train, "Label": y_train})
        df_test_raw = pd.DataFrame({"Sequence": X_test, "Label": y_test})
        
        df_train_raw.to_csv(f"{base_prefix}_train_dataset.csv", index=False)
        df_test_raw.to_csv(f"{base_prefix}_test_dataset.csv", index=False)
        print(f"[SAVED] Saved raw train/test split files: {base_prefix}_train_dataset.csv & {base_prefix}_test_dataset.csv")

        # --- Augment positive training samples only
        random.seed(base_seed)
        augmented_pos = []
        for seq in X_train[y_train == 1]:
            for _ in range(2):
                augmented_pos.append(mutate_seq(seq))

        X_train_aug = np.concatenate([X_train, augmented_pos])
        y_train_aug = np.concatenate([y_train, np.ones(len(augmented_pos))])

        # --- Save Augmented Training Set
        df_train_aug = pd.DataFrame({"Sequence": X_train_aug, "Label": y_train_aug})
        df_train_aug.to_csv(f"{base_prefix}_train_augmented_dataset.csv", index=False)

        # --- Encode sequences to One-Hot
        X_train_enc = preprocess_sequences(X_train_aug)
        X_test_enc = preprocess_sequences(X_test)
        X_train_oh = tf.keras.utils.to_categorical(X_train_enc, num_classes=num_tokens)
        X_test_oh = tf.keras.utils.to_categorical(X_test_enc, num_classes=num_tokens)

        # --- Calculate Class Weights
        cw = class_weight.compute_class_weight(
            "balanced", classes=np.unique(y_train_aug), y=y_train_aug
        )
        class_weights = dict(zip(np.unique(y_train_aug), cw))

        # --- Hyperparameter Tuning
        tuner = kt.BayesianOptimization(
            build_model,
            objective="val_accuracy",
            max_trials=10,
            seed=base_seed,
            directory="tuner_unfiltered_results",
            project_name=f"{base_prefix}_tuner"
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True
        )

        tuner.search(
            X_train_oh, y_train_aug,
            validation_split=0.2,
            epochs=30,
            batch_size=32,
            class_weight=class_weights,
            callbacks=[early_stop],
            verbose=1
        )

        # --- Retrieve best model
        best_model = tuner.get_best_models(num_models=1)[0]

        # --- Model Evaluation
        y_pred_probs = best_model.predict(X_test_oh).ravel()
        fpr, tpr, thresholds = roc_curve(y_test, y_pred_probs)
        pr, rc, _ = precision_recall_curve(y_test, y_pred_probs)
        auc_score = roc_auc_score(y_test, y_pred_probs)
        pr_auc = auc(rc, pr)
        
        optimal_threshold = thresholds[np.argmax(tpr - fpr)]
        y_pred = (y_pred_probs >= optimal_threshold).astype(int)
        
        cm = confusion_matrix(y_test, y_pred)
        tn, fp, fn, tp = cm.ravel()
        mcc = matthews_corrcoef(y_test, y_pred)

        # --- Save Individual Test Prediction Probabilities for Reviewer Audit
        df_test_preds = pd.DataFrame({
            "Sequence": X_test,
            "True_Label": y_test,
            "Predicted_Probability": y_pred_probs,
            "Predicted_Label": y_pred
        })
        df_test_preds.to_csv(f"{base_prefix}_test_predictions.csv", index=False)

        # --- Results Dictionary
        final_results = {
            "Classifier": base_prefix,
            "AUC": auc_score,
            "PR-AUC": pr_auc,
            "Optimal Threshold": float(optimal_threshold),
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred),
            "Recall": recall_score(y_test, y_pred),
            "F1": f1_score(y_test, y_pred),
            "MCC": mcc,
            "Sensitivity": tp / (tp + fn),
            "Specificity": tn / (tn + fp),
            "TP": int(tp),
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "Confusion Matrix": cm.tolist()
        }

        # --- Save Models & Results Files with NF_ Prefix
        best_model.save(f"{base_prefix}.keras")
        best_model.save(f"{base_prefix}.h5")
        joblib.dump(optimal_threshold, f"{base_prefix}_threshold.pkl")
        pd.DataFrame([final_results]).to_csv(f"{base_prefix}_results.csv", index=False)

        # --- Save ROC Curve
        plt.figure(figsize=(6, 5))
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f"AUC = {auc_score:.3f}")
        plt.plot([0, 1], [0, 1], "k--", lw=1)
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"ROC Curve: {base_prefix}")
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.savefig(f"{base_prefix}_roc.png", dpi=300)
        plt.close()

        # --- Save PR Curve
        plt.figure(figsize=(6, 5))
        plt.plot(rc, pr, color='blue', lw=2, label=f"PR-AUC = {pr_auc:.3f}")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(f"Precision-Recall Curve: {base_prefix}")
        plt.legend(loc="lower left")
        plt.tight_layout()
        plt.savefig(f"{base_prefix}_pr.png", dpi=300)
        plt.close()

        print(f"[DONE] Complete. All files generated with prefix '{base_prefix}'.\n")

Trial 10 Complete [00h 00m 51s]
val_accuracy: 0.8681591749191284

Best val_accuracy So Far: 0.893034815788269
Total elapsed time: 00h 08m 18s


C:\Users\91868\AppData\Roaming\Python\Python313\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 260ms/step


[DONE] Complete. All files generated with prefix 'NF_model_P1_N2'.

